In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# Test and Initial Visualization

In [ ]:
# Load the datasets
train_df = pd.read_csv("/kaggle/input/llm-classification-finetuning/train.csv")
test_df = pd.read_csv("/kaggle/input/llm-classification-finetuning/test.csv")
sample_submission = pd.read_csv("/kaggle/input/llm-classification-finetuning/sample_submission.csv")

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
print("Sample submission shape:", sample_submission.shape)

display(train_df.head())
display(test_df.head())
display(sample_submission.head())

# Target Distribution Exploration

In [ ]:
class_counts = train_df[['winner_model_a','winner_model_b','winner_tie']].sum()
class_counts = class_counts.rename({
    'winner_model_a': 'A wins',
    'winner_model_b': 'B wins',
    'winner_tie': 'Tie'
})

print("Class distribution (counts):")
print(class_counts)

print("\nClass distribution (percentages):")
print((class_counts / len(train_df) * 100).round(2))

# Prompt/Response Lengths

The main reason for chcking this is for model planning given the relationship between transformers and context limits.

In [ ]:
# Add basic length features
train_df['prompt_len'] = train_df['prompt'].str.split().str.len()
train_df['resp_a_len'] = train_df['response_a'].str.split().str.len()
train_df['resp_b_len'] = train_df['response_b'].str.split().str.len()

# Summary stats
print("Prompt length stats:")
print(train_df['prompt_len'].describe())

print("\nResponse A length stats:")
print(train_df['resp_a_len'].describe())

print("\nResponse B length stats:")
print(train_df['resp_b_len'].describe())


# Visualize Distribution

In [ ]:
import matplotlib.pyplot as plt

# Histograms with a reasonable cutoff to see bulk distribution
plt.figure(figsize=(12,4))
plt.hist(train_df['prompt_len'], bins=50, range=(0,200), alpha=0.7, label='Prompt')
plt.title("Prompt length distribution (0–200 words)")
plt.xlabel("Words")
plt.ylabel("Count")
plt.legend()
plt.show()

plt.figure(figsize=(12,4))
plt.hist(train_df['resp_a_len'], bins=50, range=(0,1000), alpha=0.7, label='Response A')
plt.hist(train_df['resp_b_len'], bins=50, range=(0,1000), alpha=0.7, label='Response B')
plt.title("Response length distribution (0–1000 words)")
plt.xlabel("Words")
plt.ylabel("Count")
plt.legend()
plt.show()

# Does Response Length Correlate with Winning

In [ ]:
import seaborn as sns

# Winner label as a single column
train_df['winner'] = train_df[['winner_model_a','winner_model_b','winner_tie']].idxmax(axis=1)

# Map to shorter names
train_df['winner'] = train_df['winner'].map({
    'winner_model_a': 'A',
    'winner_model_b': 'B',
    'winner_tie': 'Tie'
})

# Compare response lengths vs outcome
plt.figure(figsize=(10,6))
sns.boxplot(data=train_df, x='winner', y='resp_a_len')
plt.title("Response A length vs outcome")
plt.show()

plt.figure(figsize=(10,6))
sns.boxplot(data=train_df, x='winner', y='resp_b_len')
plt.title("Response B length vs outcome")
plt.show()

# Feature Engineering

In [ ]:
# Feature engineering
train_df['len_diff'] = (train_df['resp_a_len'] - train_df['resp_b_len']).abs()

# Encode model IDs as categorical codes
train_df['model_a_code'] = train_df['model_a'].astype('category').cat.codes
train_df['model_b_code'] = train_df['model_b'].astype('category').cat.codes

# Quick check
train_df[['prompt_len','resp_a_len','resp_b_len','len_diff','model_a_code','model_b_code']].head()

# First Baseline Model

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import log_loss
from sklearn.linear_model import LogisticRegression

# Prepare features and target
X = train_df[['prompt_len','resp_a_len','resp_b_len','len_diff','model_a_code','model_b_code']]
y = train_df['winner']

# Train/validation split
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Fit baseline logistic regression
clf = LogisticRegression(max_iter=1000, multi_class='multinomial')
clf.fit(X_train, y_train)

# Predict probabilities and compute log loss
val_probs = clf.predict_proba(X_val)
score = log_loss(y_val, val_probs)

print("Validation log loss:", score)

# Online - Classifier and Embeddings

In [ ]:
#import torch
#import pandas as pd
#import numpy as np
#from sentence_transformers import SentenceTransformer
#from sklearn.preprocessing import LabelEncoder

# Paths
#train_path = "/kaggle/input/llm-classification-finetuning/train.csv"
#save_prefix = "/kaggle/working/llm_features"

# Load training data
#train_df = pd.read_csv(train_path)

# Detect device
#device = "cuda" if torch.cuda.is_available() else "cpu"
#print("Using device:", device)

# Load embedding model (only available with internet!)
#embedder = SentenceTransformer("all-MiniLM-L6-v2", device=device)

# Encode prompt, response A, response B
#prompt_embeddings = embedder.encode(train_df["prompt"].tolist(), batch_size=64, show_progress_bar=True)
#resp_a_embeddings = embedder.encode(train_df["response_a"].tolist(), batch_size=64, show_progress_bar=True)
#resp_b_embeddings = embedder.encode(train_df["response_b"].tolist(), batch_size=64, show_progress_bar=True)

# Combine embedding features
#X_text = np.hstack([
#    prompt_embeddings,
#    resp_a_embeddings,
#    resp_b_embeddings,
#    resp_a_embeddings - resp_b_embeddings
#])

# Add numeric features
#train_df["prompt_len"] = train_df["prompt"].str.split().str.len()
#train_df["resp_a_len"] = train_df["response_a"].str.split().str.len()
#train_df["resp_b_len"] = train_df["response_b"].str.split().str.len()
#train_df["len_diff"] = (train_df["resp_a_len"] - train_df["resp_b_len"]).abs()
#train_df["model_a_code"] = train_df["model_a"].astype("category").cat.codes
#train_df["model_b_code"] = train_df["model_b"].astype("category").cat.codes

#X_numeric = train_df[["prompt_len","resp_a_len","resp_b_len","len_diff","model_a_code","model_b_code"]].values

# Final feature matrix
#X_all = np.hstack([X_text, X_numeric])

# Collapse one-hot winners into single label column
#train_df["winner"] = train_df[["winner_model_a","winner_model_b","winner_tie"]].idxmax(axis=1)
#train_df["winner"] = train_df["winner"].map({
#    "winner_model_a": "A",
#    "winner_model_b": "B",
#    "winner_tie": "Tie"
#})

# Encode target labels
#le = LabelEncoder()
#y = le.fit_transform(train_df["winner"])

# Save features + labels
#np.save(f"{save_prefix}_X.npy", X_all)
#np.save(f"{save_prefix}_y.npy", y)

#print("Saved feature arrays:")
#print(f"  {save_prefix}_X.npy shape:", X_all.shape)
#print(f"  {save_prefix}_y.npy shape:", y.shape)

In [ ]:
#import numpy as np
#import pandas as pd

# Load test set
#test_df = pd.read_csv("/kaggle/input/llm-classification-finetuning/test.csv")

# Encode prompt + responses
#test_prompt_embeddings = embedder.encode(test_df['prompt'].tolist(), batch_size=64, show_progress_bar=True)
#test_resp_a_embeddings = embedder.encode(test_df['response_a'].tolist(), batch_size=64, show_progress_bar=True)
#test_resp_b_embeddings = embedder.encode(test_df['response_b'].tolist(), batch_size=64, show_progress_bar=True)

# Build features
#X_test = np.hstack([
#    test_prompt_embeddings,
#    test_resp_a_embeddings,
#    test_resp_b_embeddings,
#    test_resp_a_embeddings - test_resp_b_embeddings
#])

# Save features + ids
#np.save("/kaggle/working/llm_features_test_X.npy", X_test)
#test_df[["id"]].to_csv("/kaggle/working/llm_test_ids.csv", index=False)

#print("Saved:")
#print("  X_test -> /kaggle/working/llm_features_test_X.npy", X_test.shape)
#print("  ids    -> /kaggle/working/llm_test_ids.csv", test_df.shape)

# Offline Dataset Run

In [ ]:
import numpy as np

# Load precomputed features
X_all = np.load("/kaggle/input/main-contest-dataset/llm_features_X.npy")
y = np.load("/kaggle/input/main-contest-dataset/llm_features_y.npy")

print("Loaded feature arrays:")
print("  X_all:", X_all.shape)
print("  y:", y.shape)

In [ ]:
X_test = np.load("/kaggle/input/dataset-test2/llm_features_test_X.npy")
test_ids = pd.read_csv("/kaggle/input/dataset-test2/llm_test_ids.csv")

# Grid Search

In [ ]:
import numpy as np
import xgboost as xgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import log_loss
import matplotlib.pyplot as plt

# Fixed params except sampling strategies
base_params = {
    "max_depth": 8,
    "eta": 0.05,
    "num_boost_round": 400
}

sampling_grid = [
    {"subsample": 0.8, "colsample_bytree": 0.8},
    {"subsample": 0.8, "colsample_bytree": 1.0},
    {"subsample": 1.0, "colsample_bytree": 0.8},
    {"subsample": 1.0, "colsample_bytree": 1.0},
]

kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = {}

for sampling in sampling_grid:
    params = {**base_params, **sampling}
    print(f"\n+ Testing hyperparameters: {params}\n")
    
    cv_scores = []
    all_curves = []

    for fold, (train_idx, val_idx) in enumerate(kf.split(X_all, y), 1):
        X_train, X_val = X_all[train_idx], X_all[val_idx]
        y_train, y_val = y[train_idx], y[val_idx]

        dtrain = xgb.DMatrix(X_train, label=y_train)
        dval = xgb.DMatrix(X_val, label=y_val)

        evals_result = {}
        booster = xgb.train(
            params={
                "objective": "multi:softprob",
                "num_class": 3,
                "eval_metric": "mlogloss",
                "tree_method": "hist",
                "device": "cuda",
                "max_depth": params["max_depth"],
                "eta": params["eta"],
                "subsample": params["subsample"],
                "colsample_bytree": params["colsample_bytree"],
            },
            dtrain=dtrain,
            num_boost_round=params["num_boost_round"],
            evals=[(dval, "val")],
            early_stopping_rounds=30,
            evals_result=evals_result,
            verbose_eval=False
        )

        val_probs = booster.predict(dval, iteration_range=(0, booster.best_iteration+1))
        score = log_loss(y_val, val_probs)
        cv_scores.append(score)

        print(f"  Fold {fold}: log loss={score:.4f}, best_iteration={booster.best_iteration}")

        all_curves.append(evals_result["val"]["mlogloss"])

    mean_score = np.mean(cv_scores)
    results[(params["subsample"], params["colsample_bytree"])] = (mean_score, all_curves)
    print(f"\n+ Mean CV log loss: {mean_score:.4f}")
    print("-"*120)

# --- Plotting ---
plt.figure(figsize=(10,7))
for (subsample, colsample), (mean_score, curves) in results.items():
    # Normalize curve lengths by cutting/padding
    max_len = max(len(c) for c in curves)
    norm_curves = []
    for c in curves:
        if len(c) < max_len:
            c = c + [c[-1]] * (max_len - len(c))  # pad with last value
        norm_curves.append(np.array(c))
    mean_curve = np.mean(norm_curves, axis=0)
    
    plt.plot(mean_curve, label=f"sub={subsample}, col={colsample}, logloss={mean_score:.4f}")

plt.xlabel("Boosting Round")
plt.ylabel("Validation Log Loss")
plt.title("Mean learning curves for subsample/colsample variations")
plt.legend()
plt.show()

# Final Training

In [ ]:
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

# Best hyperparameters from CV
final_params = {
    "objective": "multi:softprob",
    "num_class": 3,
    "eval_metric": "mlogloss",
    "tree_method": "hist",
    "device": "cuda",
    "max_depth": 8,
    "eta": 0.05,
    "subsample": 1.0,
    "colsample_bytree": 0.8
}
num_boost_round = 400

# Train/val split for monitoring
X_train, X_val, y_train, y_val = train_test_split(
    X_all, y, test_size=0.1, random_state=42, stratify=y
)

dtrain = xgb.DMatrix(X_train, label=y_train)
dval   = xgb.DMatrix(X_val, label=y_val)

evals_result = {}
booster = xgb.train(
    params=final_params,
    dtrain=dtrain,
    num_boost_round=num_boost_round,
    evals=[(dtrain, "train"), (dval, "val")],
    early_stopping_rounds=30,
    evals_result=evals_result,
    verbose_eval=50  # only print every 50 rounds
)

print(f"\nFinal model trained with best_iteration={booster.best_iteration}, "
      f"val_logloss={evals_result['val']['mlogloss'][booster.best_iteration]:.4f}")

# Save model for later use
booster.save_model("xgb_final_model.json")

# --- Plot training vs validation ---
train_curve = evals_result["train"]["mlogloss"]
val_curve   = evals_result["val"]["mlogloss"]

plt.figure(figsize=(8,6))
plt.plot(train_curve, label="Train")
plt.plot(val_curve, label="Validation")
plt.axvline(booster.best_iteration, color="red", linestyle="--", label="Best Iteration")
plt.xlabel("Boosting Round")
plt.ylabel("Log Loss")
plt.title("Final Model Training Curve")
plt.legend()
plt.show()

# Prediction and Submission

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb

# --- Load precomputed test features ---
X_test = np.load("/kaggle/input/dataset-test2/llm_features_test_X.npy")
ids = pd.read_csv("/kaggle/input/dataset-test2/llm_test_ids.csv")

print("Test shape:", X_test.shape)

# --- Reload the trained model from Kaggle working dir ---
final_booster = xgb.Booster()
final_booster.load_model("/kaggle/working/xgb_final_model.json")

# --- Predict probabilities ---
dtest = xgb.DMatrix(X_test)
test_probs = final_booster.predict(dtest)

# --- Build submission ---
submission = pd.DataFrame({
    "id": ids["id"],   # use precomputed test IDs, not test_df
    "winner_model_a": test_probs[:, 0],
    "winner_model_b": test_probs[:, 1],
    "winner_tie": test_probs[:, 2]
})

# --- Validation checks ---
# 1. Probabilities sum to 1
row_sums = submission[["winner_model_a", "winner_model_b", "winner_tie"]].sum(axis=1)
assert np.allclose(row_sums, 1.0, atol=1e-6), "Probabilities in each row must sum to 1!"

# 2. Columns match sample submission
sample_sub = pd.read_csv("/kaggle/input/llm-classification-finetuning/sample_submission.csv")
assert list(submission.columns) == list(sample_sub.columns), "Submission columns must match sample_submission.csv"

# --- Save submission ---
submission.to_csv("submission.csv", index=False)
print("\n- Submission saved as submission.csv")
print(submission.head())